# Neural Networks Basics

**Learning Objectives:** Understand the biological inspiration, mathematical foundations, and building blocks of artificial neural networks. Implement a perceptron and forward propagation from scratch using NumPy.

## 1. Biological Neuron vs Artificial Neuron

A **biological neuron** receives signals through dendrites, processes them in the cell body (soma), and transmits output via the axon through synapses.

An **artificial neuron** mimics this:
- **Inputs** ($x_1, x_2, ..., x_n$) → like dendrites
- **Weights** ($w_1, w_2, ..., w_n$) → like synaptic strength
- **Bias** ($b$) → threshold adjustment
- **Activation function** $f(\cdot)$ → like the axon firing
- **Output** ($y$) → signal passed to next layer

Mathematically: $y = f(\sum_{i=1}^{n} w_i x_i + b)$

## 2. The Perceptron Model

The **perceptron** (Rosenblatt, 1958) is the simplest neural network — a single neuron for binary classification.

- Linear combination: $z = \mathbf{w}^T \mathbf{x} + b$
- Step activation: $\hat{y} = 1 \text{ if } z \geq 0 \text{ else } 0$
- Update rule: $w_i \leftarrow w_i + \eta (y - \hat{y}) x_i$

**Limitation:** Perceptrons can only learn linearly separable patterns (Minsky & Papert, 1969).

In [ ]:
# Perceptron implementation from scratch
import numpy as np

class Perceptron:
    """Perceptron classifier."""
    def __init__(self, learning_rate=0.01, n_iters=1000):
        self.lr = learning_rate
        self.n_iters = n_iters
        self.weights = None
        self.bias = None

    def _step_function(self, z):
        return np.where(z >= 0, 1, 0)

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0

        for _ in range(self.n_iters):
            for idx in range(n_samples):
                z = np.dot(X[idx], self.weights) + self.bias
                y_pred = self._step_function(z)
                update = self.lr * (y[idx] - y_pred)
                self.weights += update * X[idx]
                self.bias += update

    def predict(self, X):
        z = np.dot(X, self.weights) + self.bias
        return self._step_function(z)

In [ ]:
# Test perceptron on synthetic data
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X, y = make_classification(n_samples=200, n_features=2, n_redundant=0,
                           n_clusters_per_class=1, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

p = Perceptron(learning_rate=0.01, n_iters=500)
p.fit(X_train, y_train)
y_pred = p.predict(X_test)
print(f"Perceptron accuracy: {accuracy_score(y_test, y_pred):.3f}")

## 3. Activation Functions

Activation functions introduce **non-linearity**, enabling NNs to learn complex patterns.

In [ ]:
# Visualize activation functions
import matplotlib.pyplot as plt

def sigmoid(x): return 1 / (1 + np.exp(-x))
def tanh(x): return np.tanh(x)
def relu(x): return np.maximum(0, x)
def leaky_relu(x, alpha=0.01): return np.where(x > 0, x, alpha * x)

x = np.linspace(-10, 10, 200)
activations = {
    'Sigmoid': sigmoid(x),
    'Tanh': tanh(x),
    'ReLU': relu(x),
    'Leaky ReLU': leaky_relu(x)
}

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for ax, (name, y) in zip(axes.flat, activations.items()):
    ax.plot(x, y, linewidth=2)
    ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
    ax.axvline(0, color='gray', linestyle='--', alpha=0.5)
    ax.set_title(name, fontsize=14)
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Activation Function Comparison

| Function | Range | Output | Pros | Cons |
|----------|-------|--------|------|------|
| Sigmoid | (0, 1) | Probabilistic | Smooth, differentiable | Vanishing gradient, not zero-centered |
| Tanh | (-1, 1) | Zero-centered | Stronger gradients | Vanishing gradient still possible |
| ReLU | [0, ∞) | Sparse | No vanishing gradient, fast | Dying ReLU (dead neurons) |
| Leaky ReLU | (-∞, ∞) | Sparse | Fixes dying ReLU | Slightly more computation |
| Softmax | (0, 1) | Probability distribution | Multi-class output | Depends on all outputs |

## 4. Forward Propagation

Forward propagation computes the output of a neural network layer by layer:

$z^{[l]} = W^{[l]} a^{[l-1]} + b^{[l]}$

$a^{[l]} = g^{[l]}(z^{[l]})$  where $g^{[l]}$ is the activation at layer $l$

In [ ]:
# Manual forward propagation for a 3-layer network
np.random.seed(42)

def initialize_parameters(layer_dims):
    params = {}
    for l in range(1, len(layer_dims)):
        params[f'W{l}'] = np.random.randn(layer_dims[l], layer_dims[l-1]) * 0.01
        params[f'b{l}'] = np.zeros((layer_dims[l], 1))
    return params

def forward_propagation(X, params):
    cache = {'A0': X}
    L = len(params) // 2  # number of layers
    for l in range(1, L + 1):
        Z = params[f'W{l}'] @ cache[f'A{l-1}'] + params[f'b{l}']
        # ReLU for hidden layers, sigmoid for output
        A = np.maximum(Z, 0) if l < L else 1 / (1 + np.exp(-Z))
        cache[f'Z{l}'] = Z
        cache[f'A{l}'] = A
    return cache[f'A{L}'], cache

# Demo: forward pass on random data
X_sample = np.random.randn(3, 5)  # 5 samples, 3 features
params = initialize_parameters([3, 4, 4, 1])
y_hat, cache = forward_propagation(X_sample, params)
print(f"Input shape: {X_sample.shape}")
print(f"Output shape: {y_hat.shape}")
print(f"Predictions:\n{y_hat}")

## 5. Loss Functions

Loss functions measure the error between predictions and true values.

**Mean Squared Error (MSE)** — for regression:
$\mathcal{L}_{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$

**Binary Cross-Entropy** — for binary classification:
$\mathcal{L}_{BCE} = -\frac{1}{n} \sum_{i=1}^{n} [y_i \log(\hat{y}_i) + (1 - y_i) \log(1 - \hat{y}_i)]$

**Categorical Cross-Entropy** — for multi-class classification:
$\mathcal{L}_{CCE} = -\sum_{i=1}^{n} \sum_{c=1}^{C} y_{i,c} \log(\hat{y}_{i,c})$

In [ ]:
# Loss function implementations
def mse_loss(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

def binary_cross_entropy(y_true, y_pred):
    eps = 1e-15  # prevent log(0)
    y_pred = np.clip(y_pred, eps, 1 - eps)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

def categorical_cross_entropy(y_true, y_pred):
    eps = 1e-15
    y_pred = np.clip(y_pred, eps, 1 - eps)
    return -np.mean(np.sum(y_true * np.log(y_pred), axis=1))

# Demo
y_true_bin = np.array([1, 0, 1, 0])
y_pred_bin = np.array([0.9, 0.1, 0.8, 0.3])
print(f"BCE: {binary_cross_entropy(y_true_bin, y_pred_bin):.4f}")

y_true_oh = np.array([[1,0,0], [0,1,0], [0,0,1]])
y_pred_oh = np.array([[0.7,0.2,0.1], [0.1,0.8,0.1], [0.2,0.3,0.5]])
print(f"CCE: {categorical_cross_entropy(y_true_oh, y_pred_oh):.4f}")

## 6. Gradient Descent Intuition

Gradient descent iteratively adjusts parameters to minimize the loss:

$\theta_{t+1} = \theta_t - \eta \nabla_\theta \mathcal{L}(\theta_t)$

Where:
- $\theta$ = parameters (weights & biases)
- $\eta$ = **learning rate** — how big a step we take
- $\nabla_\theta \mathcal{L}$ = gradient of loss w.r.t. parameters

**Learning rate** is the most important hyperparameter:
- Too **large**: diverges, loss explodes
- Too **small**: converges very slowly, gets stuck
- Just **right**: smooth convergence to minimum

In [ ]:
# Visualize gradient descent with different learning rates
def f(x): return x**2 + 3*x + 2
def df(x): return 2*x + 3

def gradient_descent(lr, epochs=30):
    x = 6.0  # starting point
    history = [x]
    for _ in range(epochs):
        x = x - lr * df(x)
        history.append(x)
    return history

lrs = [0.01, 0.1, 0.5, 1.0]
x_plot = np.linspace(-7, 7, 200)
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for ax, lr in zip(axes.flat, lrs):
    hist = gradient_descent(lr)
    ax.plot(x_plot, f(x_plot), 'b-', label='Loss surface')
    ax.scatter(hist, f(np.array(hist)), c='r', s=30, zorder=5)
    ax.plot(hist, f(np.array(hist)), 'r--', alpha=0.5)
    ax.set_title(f'LR = {lr}')
    ax.axvline(-1.5, color='green', linestyle=':', label='Minimum')
    ax.legend()
plt.tight_layout()
plt.show()

## 7. Building a Neural Network from Scratch

Let's build a complete 2-layer NN using only NumPy with forward pass, loss computation, and (conceptually) gradient descent.

This is a **binary classification** network with architecture: `input -> hidden(ReLU) -> output(Sigmoid)`.

In [ ]:
class TwoLayerNN:
    def __init__(self, input_size, hidden_size, lr=0.01):
        self.lr = lr
        self.W1 = np.random.randn(hidden_size, input_size) * 0.01
        self.b1 = np.zeros((hidden_size, 1))
        self.W2 = np.random.randn(1, hidden_size) * 0.01
        self.b2 = np.zeros((1, 1))

    def forward(self, X):
        self.Z1 = self.W1 @ X + self.b1
        self.A1 = np.maximum(self.Z1, 0)  # ReLU
        self.Z2 = self.W2 @ self.A1 + self.b2
        self.A2 = 1 / (1 + np.exp(-self.Z2))  # Sigmoid
        return self.A2

    def compute_loss(self, y_true, y_pred):
        eps = 1e-15
        y_pred = np.clip(y_pred, eps, 1 - eps)
        return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

    def backward(self, X, y_true):
        m = X.shape[1]
        dZ2 = self.A2 - y_true  # gradient of BCE + sigmoid
        dW2 = (1 / m) * dZ2 @ self.A1.T
        db2 = (1 / m) * np.sum(dZ2, axis=1, keepdims=True)
        dA1 = self.W2.T @ dZ2
        dZ1 = dA1 * (self.Z1 > 0)  # ReLU derivative
        dW1 = (1 / m) * dZ1 @ X.T
        db1 = (1 / m) * np.sum(dZ1, axis=1, keepdims=True)
        return {'dW1': dW1, 'db1': db1, 'dW2': dW2, 'db2': db2}

    def update(self, grads):
        self.W1 -= self.lr * grads['dW1']
        self.b1 -= self.lr * grads['db1']
        self.W2 -= self.lr * grads['dW2']
        self.b2 -= self.lr * grads['db2']

In [ ]:
# Train the two-layer NN on synthetic data
from sklearn.datasets import make_moons

X_moon, y_moon = make_moons(n_samples=300, noise=0.2, random_state=42)
X_moon = X_moon.T  # shape (2, 300)
y_moon = y_moon.reshape(1, -1)  # shape (1, 300)

model = TwoLayerNN(input_size=2, hidden_size=10, lr=0.5)

losses = []
for i in range(1000):
    y_pred = model.forward(X_moon)
    loss = model.compute_loss(y_moon, y_pred)
    losses.append(loss)
    grads = model.backward(X_moon, y_moon)
    model.update(grads)
    if i % 200 == 0:
        print(f"Epoch {i}, Loss: {loss:.4f}")

plt.figure(figsize=(8, 4))
plt.plot(losses)
plt.title('Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(alpha=0.3)
plt.show()

## 8. Interview Problems

### Problem 1: Implement Perceptron from Scratch

**Task:** Complete the Perceptron implementation above and test it on the Iris dataset (binary: setosa vs. non-setosa).

### Problem 2: Implement Forward Propagation Manually

**Task:** Write a function `manual_forward(X, weights, biases)` that takes a batch of inputs and layer parameters, and computes the forward pass through a 4-layer network (3 hidden with ReLU, output with softmax). Return both the final output and all intermediate activations.

### Problem 3: Activation Function Derivatives

**Task:** Implement the derivatives of sigmoid, tanh, ReLU, and Leaky ReLU. Verify numerically using finite differences.

In [ ]:
# Solution: Problem 2 — Manual forward propagation with softmax output
def softmax(z):
    exp_z = np.exp(z - np.max(z, axis=0, keepdims=True))  # numerical stability
    return exp_z / np.sum(exp_z, axis=0, keepdims=True)

def manual_forward(X, weights, biases):
    """
    weights: list of [W1, W2, W3, W4]
    biases: list of [b1, b2, b3, b4]
    """
    cache = {'A0': X}
    L = len(weights)
    for l in range(L):
        Z = weights[l] @ cache[f'A{l}'] + biases[l]
        if l < L - 1:
            A = np.maximum(Z, 0)  # ReLU hidden layers
        else:
            A = softmax(Z)  # softmax output
        cache[f'Z{l+1}'] = Z
        cache[f'A{l+1}'] = A
    return cache[f'A{L}'], cache

# Quick test
W = [np.random.randn(5, 10), np.random.randn(3, 5), np.random.randn(3, 3), np.random.randn(4, 3)]
b = [np.zeros((5, 1)), np.zeros((3, 1)), np.zeros((3, 1)), np.zeros((4, 1))]
X_test = np.random.randn(10, 20)
out, cache = manual_forward(X_test, W, b)
print(f"Output shape: {out.shape}")
print(f"Output sums to 1 (per sample): {np.allclose(np.sum(out, axis=0), 1)}")

In [ ]:
# Solution: Problem 3 — Activation derivatives with numerical verification
def sigmoid_derivative(z):
    s = 1 / (1 + np.exp(-z))
    return s * (1 - s)

def tanh_derivative(z):
    t = np.tanh(z)
    return 1 - t**2

def relu_derivative(z):
    return (z > 0).astype(float)

def leaky_relu_derivative(z, alpha=0.01):
    return np.where(z > 0, 1.0, alpha)

# Numerical derivative check (finite differences)
def numerical_derivative(f, z, eps=1e-7):
    return (f(z + eps) - f(z - eps)) / (2 * eps)

z_test = np.array([-2.0, -1.0, 0.0, 1.0, 2.0])
print("Function | z=-2  z=-1  z=0   z=1   z=2")
for name, func, dfunc in [
    ('Sigmoid', sigmoid, sigmoid_derivative),
    ('Tanh', tanh, tanh_derivative),
    ('ReLU', relu, relu_derivative)
]:
    analytic = dfunc(z_test)
    numeric = numerical_derivative(func, z_test)
    match = np.allclose(analytic, numeric, atol=1e-5)
    print(f"{name:8s} | {match!s}")

## Summary

- **Neurons** compute weighted sums followed by non-linear activations
- **Perceptron** is the simplest neuron — limited to linear boundaries
- **Activation functions** like ReLU and Tanh enable deep networks to learn complex patterns
- **Forward propagation** passes data through layers to produce predictions
- **Loss functions** quantify prediction error — choose based on task (regression vs. classification)
- **Gradient descent** iteratively minimizes loss by updating parameters in the negative gradient direction
- **Learning rate** controls step size and is critical for convergence